# Kilosort Alignment Verification Test

This notebook verifies that kilosort spike times are correctly aligned by comparing them with LFP traces from `oe_rec.get_data()`.

## Test Strategy

1. Load kilosort spike data (aligned to OE recording start)
2. Load LFP traces for channels with known spiking activity (18, 17, 9, 10)
3. Overlay spike times as vertical lines on LFP traces
4. Verify that spikes align with spike waveforms in the LFP traces

If alignment is correct, spikes should appear at the same times as the spike waveforms in the LFP.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add project to path if needed
project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing.kilosort_alignment_verification import (
    verify_spike_alignment_with_lfp,
    verify_spike_alignment_summary
)

## 1. Initialize BlockSync Object

In [ ]:
# Configuration
experiment_path = Path(r"D:\sample_data_for_eye_repo")
animal = "PV_208"
experiment_date = "2025_12_14"
block_num = "019"

# Channel mapping
channeldict = {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"}

# Create BlockSync object
block = BlockSync(
    animal_call=animal,
    experiment_date=experiment_date,
    block_num=block_num,
    path_to_animal_folder=str(experiment_path),
    channeldict=channeldict
)

print(f"Block path: {block.block_path}")
print(f"OE path: {block.oe_path}")
print(f"Sample rate: {block.sample_rate} Hz")
print(f"OE recording start time: {block.oe_rec.globalStartTime_ms:.2f} ms")
print(f"OE recording duration: {block.oe_rec.recordingDuration_ms:.2f} ms")

## 2. Generate Alignment Summary

In [ ]:
# Check alignment for first 10 seconds
lfp_channels = [18, 17, 9, 10]  # Channels with known spiking activity
time_range_ms = (0, 10000)  # First 10 seconds

summary = verify_spike_alignment_summary(
    block=block,
    lfp_channels=lfp_channels,
    time_range_ms=time_range_ms,
    filter_good_only=True
)

print("Alignment Summary:")
print(f"  Time range: {summary['time_range_ms']} ms")
print(f"  LFP channels: {summary['lfp_channels']}")
print(f"  Number of spikes in range: {summary['n_spikes_in_range']:,}")
print(f"  Number of clusters: {summary['n_clusters']}")
print(f"\nSpike time range:")
print(f"  Min: {summary['spike_time_range']['min']:.2f} ms")
print(f"  Max: {summary['spike_time_range']['max']:.2f} ms")
print(f"\nLFP time range:")
print(f"  Min: {summary['lfp_time_range']['min']:.2f} ms")
print(f"  Max: {summary['lfp_time_range']['max']:.2f} ms")
print(f"\nTime alignment check:")
print(f"  Spike min - LFP min: {summary['time_alignment_check']['spike_min_vs_lfp_min']:.2f} ms")
print(f"  Spike max - LFP max: {summary['time_alignment_check']['spike_max_vs_lfp_max']:.2f} ms")
print(f"\nOE Recording Info:")
print(f"  Global start time: {summary['oe_recording_info']['globalStartTime_ms']:.2f} ms")
print(f"  Recording duration: {summary['oe_recording_info']['recordingDuration_ms']:.2f} ms")
print(f"  Sample rate: {summary['oe_recording_info']['sample_rate']:.0f} Hz")
print(f"\nKilosort Info:")
print(f"  Kilosort sample rate: {summary['kilosort_info']['kilosort_sample_rate']:.0f} Hz")
print(f"  Total spikes: {summary['kilosort_info']['n_spikes_total']:,}")
print(f"  Good spikes: {summary['kilosort_info']['n_spikes_returned']:,}")
print(f"  Good clusters: {summary['kilosort_info']['n_good_clusters']}")

## 3. Visual Verification: Overlay Spikes on LFP Traces

This creates plots showing LFP traces with kilosort spikes overlaid as red vertical lines.
If alignment is correct, spikes should appear at the same times as spike waveforms in the LFP.

In [ ]:
# Plot full 10-second window (may be slow with many spikes)
figures = verify_spike_alignment_with_lfp(
    block=block,
    lfp_channels=lfp_channels,
    time_range_ms=(0, 10000),  # First 10 seconds
    filter_good_only=True,
    width=1400,
    height_per_channel=200,
    downsample_lfp=1,  # No downsampling for full resolution
    to_browser=True
)

## 4. Detailed View: Zoom into Specific Time Window

Zoom into a shorter time window (e.g., 1 second) to see individual spikes more clearly.

In [ ]:
# Plot a 1-second window for detailed inspection
figures_detail = verify_spike_alignment_with_lfp(
    block=block,
    lfp_channels=lfp_channels,
    time_range_ms=(1000, 2000),  # 1-2 seconds
    filter_good_only=True,
    width=1400,
    height_per_channel=250,
    downsample_lfp=1,
    to_browser=True
)

## 5. Check Specific Clusters

If you want to verify alignment for specific clusters, you can filter by cluster IDs.

In [ ]:
from eye_tracking_system_tools.preprocessing.kilosort_loader import get_good_cluster_ids

# Get good cluster IDs
kilosort_path = block.oe_path / 'spikeSorting' / 'kilosort'
good_clusters = get_good_cluster_ids(kilosort_path)
print(f"Good cluster IDs: {good_clusters}")

# Plot with specific clusters (e.g., first 3 good clusters)
if len(good_clusters) >= 3:
    clusters_to_check = good_clusters[:3]
    print(f"\nChecking clusters: {clusters_to_check}")
    
    figures_clusters = verify_spike_alignment_with_lfp(
        block=block,
        lfp_channels=lfp_channels,
        time_range_ms=(1000, 2000),
        filter_good_only=False,  # We'll filter manually
        cluster_ids=clusters_to_check,
        width=1400,
        height_per_channel=250,
        to_browser=True
    )

## 6. Alignment Verification Results

### Expected Results:

1. **Time alignment**: Spike times should match LFP timestamps (within sampling precision)
2. **Visual alignment**: Red spike markers should appear at the same times as spike waveforms in the LFP traces
3. **Time range**: Spike times should be within the LFP time range

### If alignment is correct:
- Spikes will appear as red vertical lines/dots aligned with spike waveforms in the LFP
- Spike times will match LFP timestamps
- No systematic offset between spike times and LFP times

### If alignment is incorrect:
- Spikes will appear offset from spike waveforms
- Systematic time offset between spike times and LFP times
- Check kilosort sample rate and OE sample rate match expectations